# SiFi Devices → OSC: Real-Time Physiological Signals
### MishMash Hackathon 2026 — Getting Started Guide

---

## About SiFi Labs

[SiFi Labs](https://sifilabs.com) builds wearable biosensors designed for researchers, artists and engineers who want to interface with the human body. The goal is to make high-quality physiological data as easy to work with as any other sensor — plug in, configure, stream.

The device you have in front of you is the **SiFi Band**. It is a wrist-worn wireless armband that measures 8-channel muscle electrical activity (EMG) and motion data over Bluetooth. Under the hood it runs a mixed-signal analog front-end with onboard signal processing; you do not need to worry about any of that — the Python library handles everything.

### What the SiFi Band can measure

| Signal | Full name | What it captures | Default sampling rate |
|--------|-----------|-----------------|----------------------|
| **EMG** | Electromyography (8-channel) | Muscle electrical activity — contractions, gestures, fatigue | 1600 Hz |
| **ECG** | Electrocardiography | Heart electrical activity — beat timing, rhythm, HRV | 500 Hz |
| **PPG** | Photoplethysmography | Blood volume changes — heart rate, SpO₂, perfusion | 50 Hz |
| **IMU** | Inertial Measurement Unit | Acceleration + orientation (quaternion) — movement, posture | 100 Hz |
| **ST** | Skin Temperature | Skin surface temperature | 1Hz |

You can enable any combination of these channels simultaneously.

### How the software stack works for this workshop

```
 SiFi Band (Bluetooth)
       │
       ▼
 sifi-bridge CLI  ← downloaded automatically by the Python library
       │ stdin / stdout
       ▼
 sifi_bridge_py   ← the Python library you will use
       │
       ▼
 your code  ——OSC/UDP——►  Max/MSP, TouchDesigner, Pure Data, Unity …
```

`sifi_bridge_py` launches a small CLI binary as a subprocess and communicates with it over stdin/stdout. The library handles downloading the right binary for your platform — you never touch it directly.

---

## Table of Contents
1. [Installation](#1.-Installation)
2. [Scan and Connect to Your Device](#2.-Scan-and-Connect-to-Your-Device)
3. [Configure Channels & Filters](#3.-Configure-Channels-&-Filters)
4. [Read a Single Packet](#4.-Read-a-Single-Packet)
5. [Real-Time OSC Streaming](#5.-Real-Time-OSC-Streaming)
6. [Receiving OSC in Your App](#6.-Receiving-OSC-in-Your-App)
7. [Verify OSC: Live Plot from OSC Stream](#7.-Verify-OSC:-Live-Plot-from-OSC-Stream)
8. [OSC Address Reference](#8.-OSC-Address-Reference)


---
## 1. Installation

Install the two libraries we need — `sifi_bridge_py` to talk to the device, and `python-osc` to send OSC messages. We will also use matplotlib to visualize the data
For a valid Python installation, we strongly recommend using Python 3.12

In [ ]:
%pip install sifi_bridge_py python-osc matplotlib ipympl --quiet

> **Note:** On the very first run, `sifi_bridge_py` will automatically download the SiFi Bridge CLI executable (~10 MB). This only happens once.

> **Note 2:** On the first connection, you might have to pair your device with your computer.

In [ ]:
# Check installed package versions
import sys
import importlib.metadata

# 1. Print Python version
print(f"Python Version: {sys.version}")
print("-" * 30)

# 2. Print Library versions
packages = ["sifi-bridge-py", "python-osc", "matplotlib", "ipympl"]

for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}: [Not Found]")

---
## 2. Scan and Connect to Your Device

At the hackathon there are **multiple SiFi Bands in the room**. Before connecting, scan for nearby devices to find your device's MAC address, then connect to it explicitly so you do not accidentally grab someone else's sensor.

### Step 1 — Scan for nearby SiFi Bands


In [ ]:
from sifi_bridge_py import SifiBridge, DeviceType, ListSources

sb = SifiBridge()

print("Scanning for BLE devices... (takes a few seconds)")
devices = sb.list_devices(source=ListSources.BLE)

print(f"\nFound {len(devices)} device(s):")
for i, d in enumerate(devices):
    print(f"  [{i}] {d}")

You will see output like:
```
Found 11 device(s):
  [0] {'name': 'P mesh', 'id': 'D8:5E:88:EA:27:39'}
  [1] {'name': 'S39 FCEE LE', 'id': '69:91:8C:19:8D:E8'}
  [2] {'name': 'Office', 'id': 'DA:6D:33:9C:7F:0B'}
  [3] {'name': 'BioArmband_v1_1', 'id': 'FF:62:16:E4:C8:99'}
  [4] {'name': 'P mesh', 'id': 'DB:84:6E:02:D0:ED'}
```

The MAC address printed on the label of your SiFi Band is your device's identifier. Copy it and paste it below.

> **macOS note:** macOS does not expose raw MAC addresses over Bluetooth — you will see a UUID string instead (e.g. `12345678-ABCD-...`). Use that UUID the same way.

### Step 2 — Connect to your specific device


In [ ]:
# ── Set this to YOUR device's MAC address from the scan above ─────────────────
MY_DEVICE_MAC = "C7:ED:F6:CF:B7:BE"   # <── change this!

print(f"Connecting to {MY_DEVICE_MAC}...")
while not sb.connect(MY_DEVICE_MAC):
    print("  Not found yet, retrying...")

print("Connected!")
print(sb.show())

> **No MAC address on your device?** You can also connect to the first SiFi Labs device available with:
> ```python
> sb.connect()
> ```
> This is fine when you are alone in the room, but at the hackathon use the MAC address to be sure.


---
## 3. Configure Channels & Filters

Each sensor has its own `configure_*` function that sets both the **sampling rate** and the **onboard filters** in one call. Passing `state=True` enables the sensor; `state=False` disables it.

You only need to call the `configure_*` functions for the sensors you want to use. All others default to off.

After configuring, call `set_filters(enable=True)` once to confirm that the onboard filter chain is active.

---
### EMG — 8-channel muscle electrical activity (default 1600 Hz)


In [ ]:
# EMG captures the electrical signal generated when muscles contract.
# Great for gesture recognition, effort detection, biofeedback, muscle fatigue.
# The SiFi Band streams 8 simultaneous EMG channels.
#
# fs:         Sampling rate in Hz. Options: 500, 1000, 1600. Default 1600.
# dc_notch:   Remove DC offset (almost always True).
# mains_notch: Remove power-line interference. 50 Hz in Europe, 60 Hz in North America, None to disable.
# flo / fhi:  Bandpass filter cutoffs in Hz. 20–450 Hz is the clinical standard.

sb.configure_emg(
    state=True,
    fs=1600,
    dc_notch=True,
    mains_notch=50,   # use 60 if in North America
    bandpass=True,
    flo=20,
    fhi=450,
)
print("EMG configured — 1600 Hz, 20–450 Hz bandpass, 50 Hz notch")


### ECG — Heart electrical activity (default 500 Hz)

In [ ]:
# ECG captures the electrical activity of the heart.
# Use it for heart rate, heart rate variability (HRV), beat detection.
#
# fs:         Sampling rate. Options include 500, 1000, 2000 Hz. Default 500.
# dc_notch:   Remove baseline wander / DC drift.
# mains_notch: Same as EMG — 50 or 60 Hz.
# flo / fhi:  0–30 Hz captures all ECG features (P, QRS, T waves).

sb.configure_ecg(
    state=True,
    fs=500,
    dc_notch=True,
    mains_notch=50,
    bandpass=True,
    flo=0,
    fhi=30,
)
print("ECG configured — 500 Hz, 0–30 Hz bandpass, 50 Hz notch")

### PPG — Blood volume / heart rate (default 50 Hz)

In [ ]:
from sifi_bridge_py import PpgSensitivity

# PPG uses LEDs and a photodiode to measure blood volume changes under the skin.
# Useful for heart rate, SpO2, and perfusion index.
#
# The SiFi Band has 4 LED channels: IR, Red, Green, Blue.
# Data keys in packets: "ir", "r", "g", "b"
#
# fs:   Sampling rate. Options: 50, 100 Hz. Default 50.
# ir/red/green/blue: LED current in mA (1–50). Higher = more light = stronger signal but more power.
# sens: Photodiode gain. LOW | MEDIUM | HIGH | MAX.
#       Increase sensitivity if the signal looks flat; decrease if it saturates.
# avg:  Averaging factor. Higher = smoother but require highest acquisition frequencies. 

sb.configure_ppg(
    state=True,
    fs=50,
    ir=9,
    red=9,
    green=9,
    blue=9,
    sens=PpgSensitivity.MEDIUM,
    avg=8,
)
print("PPG configured — 50 Hz, all LEDs at 9 mA, MEDIUM sensitivity")


### IMU — Motion & orientation (default 50 Hz)

In [ ]:
# IMU measures both linear acceleration and orientation.
# Data keys in packets:
#   Quaternion:   "qw", "qx", "qy", "qz"
#   Acceleration: "ax", "ay", "az"  (in g)
#
# fs:          Sampling rate. Options: 50, 100 Hz. Default 100.

sb.configure_imu(
    state=True,
    fs=100
)
print("IMU configured — 50 Hz")

### Putting it together — activate your chosen channels

The `configure_*` calls above both configure and enable each sensor. Use `configure_sensors()` to quickly switch sensors on or off without touching their filter settings. Then call `set_filters()` once to confirm the onboard filter chain is active.

In [ ]:
# ── Quick enable/disable without changing filter config ───────────────────────
# Edit this to match what you actually want to stream.
sb.configure_sensors(
    emg=True,
    ecg=False,
    eda=False, # EDA (electrodermal activity) is not yet implemented with the current firmware version of the SiFi Band, but will be in a future update.
    imu=True,
    ppg=False,
)

# Confirm onboard filters are active for all enabled channels
sb.set_filters(enable=True)
sb.set_low_latency_mode(on=False)  # Low-latency mode is not yet implemented with the current firmware version of the SiFi Band, but will be in a future update.

print("Channels set. Ready to stream.")

> **Sampling rate tip:** The `sample_rate` field that appears in data packets reflects the rate as experienced by the computer — it can fluctuate with Bluetooth jitter and system load. For any timing or frequency analysis always use the nominal rates you configured above (`fs` parameter), not the value from the packet.

---
## 4. Read a Single Packet

Let's start acquisition and read one packet to see exactly what the data looks like before building the full pipeline.

In [ ]:
import json

sb.start()  # begin streaming from the device

# get_data() blocks until the next packet arrives — usually a few milliseconds
start_packet = sb.get_data() # The first packet correspond to the start packet
packet = sb.get_data() # The second packet correspond to the first data packet

sb.stop()   # stop streaming

print("Start packet received:")
print(json.dumps(start_packet, indent=2))
print("Data packet received:")
print(json.dumps(packet, indent=2))

Key things to note:
- **`packet_type`** — which sensor: `"emg"`, `"ecg"`, `"eda"`, `"imu"`, `"ppg"`, `"temperature"`
- **`data`** — always a dict. EMG/ECG/EDA contain a burst of samples. IMU has individual keys per axis (`qw`, `qx`, `qy`, `qz`, `ax`, `ay`, `az`). PPG has short channel names (`ir`, `r`, `g`, `b`).
- **`sample_rate`** — reflects Bluetooth throughput, not the true acquisition rate. Use the `fs` you configured for any timing calculations.
- When multiple channels are active, you receive packets of different types interleaved — distinguish them by `packet_type`.

> We will now disconnect the SiFi Band and continue with a self-contained example for Real-Time OSC streaming


In [ ]:
sb.disconnect()

---
## 5. Real-Time OSC Streaming

### What is OSC?

**Open Sound Control (OSC)** is a network protocol that sends typed messages over UDP. Every message has:
- An **address** like `/sifi/emg/0` — a hierarchical path that identifies the data
- One or more **values** — floats, ints, strings…

Any OSC-capable software can receive these messages: Max/MSP, TouchDesigner, Pure Data, SuperCollider, Unity, p5.js, and hundreds of others.

### OSC address scheme

```
/sifi/emg/sample_rate  → float             (sent once at start, Hz)
/sifi/imu/sample_rate  → float             (sent once at start, Hz)

/sifi/emg/0            → float, float, …   (burst of EMG samples ch 0, mV)
/sifi/emg/1            → float, float, …   (burst of EMG samples ch 1, mV)
...
/sifi/emg/7            → float, float, …   (burst of EMG samples ch 7, mV)
/sifi/imu/accel        → ax, ay, az        (one sample per packet, g)
/sifi/imu/quat         → qw, qx, qy, qz   (one sample per packet)
/sifi/temperature      → float             (°C)
```

### The streamer

The cell below runs the full pipeline:

```
SiFi Band → sifi_bridge_py (background thread) → OSC/UDP → your app
```

A background thread handles the blocking device reads so the main thread stays responsive.

**Change `OSC_IP` and `OSC_PORT` to match your receiving application.**


In [ ]:
import threading
import time
import collections
from pythonosc import udp_client
from sifi_bridge_py import SifiBridge, PpgSensitivity

# =============================================================================
# CONFIGURATION  <- the only section you need to edit
# =============================================================================

MY_DEVICE_MAC = "C7:ED:F6:CF:B7:BE"   # MAC address from the scan in section 2

OSC_IP   = "127.0.0.1"  # IP of the machine running your OSC app.
                         # "127.0.0.1" = this computer.
                         # Use a LAN IP (e.g. "192.168.1.42") for another machine.
OSC_PORT = 9000          # Must match the port your OSC app listens on.

DURATION = None          # Seconds to stream. None = run until you press Ctrl+C.

# Nominal rates (Hz) matching the configure_* calls you made in section 3.
# Sent once to the receiver at startup; also used in the live rate display.
SAMPLE_RATES = {
    "emg_armband": 1600,  # SiFi Band - 8-channel EMG armband
    "imu":          100,
}

# Leave all other channels False unless you configured them in section 3.
ACTIVE_CHANNELS = {
    "emg_armband": True,
    "imu":         True,
}


# =============================================================================
# RATE TRACKER
#
# Keeps a rolling 2-second window of (wall_time, n_samples) events and reports
# the actual sample rate in Hz. Use this to spot Bluetooth drop-outs: a healthy
# stream stays within a few % of the nominal rate defined above.
# =============================================================================

class RateTracker:
    def __init__(self, window=2.0):
        self.window  = window
        self._events = collections.deque()  # (wall_time, n_samples)

    def add(self, n=1):
        # Record that n new samples just arrived.
        now = time.time()
        self._events.append((now, n))
        cutoff = now - self.window
        while self._events and self._events[0][0] < cutoff:
            self._events.popleft()

    def rate(self):
        # Return samples/second averaged over the last window seconds.
        if len(self._events) < 2:
            return 0.0
        now     = time.time()
        total   = sum(n for t, n in self._events if t >= now - self.window)
        elapsed = now - self._events[0][0]
        return total / elapsed if elapsed > 0 else 0.0


# One tracker per active channel
rate_trackers = {ch: RateTracker() for ch, on in ACTIVE_CHANNELS.items() if on}

# SiFi Band armband channel keys, in channel order (emg0 = channel 0, etc.)
ARMBAND_KEYS = ("emg0", "emg1", "emg2", "emg3", "emg4", "emg5", "emg6", "emg7")

def packet_to_osc(packet, osc):
    ptype = packet.get("packet_type")
    data  = packet.get("data", {})

    # -- SiFi Band armband: 8 channels, each a burst ---------------------------
    if ptype == "emg_armband":
        n_armband = 0
        for i, key in enumerate(ARMBAND_KEYS):
            samples = data.get(key, [])
            if samples:
                osc.send_message(f"/sifi/emg/{i}", [float(v) for v in samples])
                n_armband = max(n_armband, len(samples))
        if n_armband and "emg_armband" in rate_trackers:
            rate_trackers["emg_armband"].add(n_armband)

    # -- PPG: four LED colour channels, each a burst ---------------------------
    elif ptype == "ppg":
        n_ppg = 0
        for ch in ("ir", "r", "g", "b"):
            vals = data.get(ch, [])
            if vals:
                osc.send_message(f"/sifi/ppg/{ch}", [float(v) for v in vals])
                n_ppg = max(n_ppg, len(vals))
        if n_ppg and "ppg" in rate_trackers:
            rate_trackers["ppg"].add(n_ppg)

    # -- IMU: multi-axis burst, one OSC message per sample ---------------------
    elif ptype == "imu":
        n_imu = 0
        try:
            for ax, ay, az in zip(data["ax"], data["ay"], data["az"]):
                osc.send_message("/sifi/imu/accel", [float(ax), float(ay), float(az)])
            n_imu = len(data["ax"])
        except KeyError:
            pass
        try:
            for qw, qx, qy, qz in zip(data["qw"], data["qx"], data["qy"], data["qz"]):
                osc.send_message("/sifi/imu/quat", [float(qw), float(qx), float(qy), float(qz)])
        except KeyError:
            pass
        if n_imu and "imu" in rate_trackers:
            rate_trackers["imu"].add(n_imu)
            
    elif ptype == "ecg":
        samples = data.get(ptype, [])
        if samples:
            osc.send_message(f"/sifi/{ptype}", [float(v) for v in samples])
            if ptype in rate_trackers:
                rate_trackers[ptype].add(len(samples))
                
    # -- Temperature: single scalar --------------------------------------------
    elif ptype == "temperature":
        val = data.get("temperature")
        if val is not None:
            temp = val[0] if isinstance(val, list) else val
            if temp is not None:
                osc.send_message("/sifi/temperature", float(temp))


# =============================================================================
# ACQUISITION THREAD
#
# sb.get_data() is a blocking call that waits until the next BLE packet arrives.
# Running it in a background thread lets the main thread print the status line
# without introducing delays into the data path.
# =============================================================================

stop_event   = threading.Event()
packet_count = 0

def acquisition_thread(sb, osc):
    global packet_count
    while not stop_event.is_set():
        try:
            packet = sb.get_data()
            if packet:
                packet_to_osc(packet, osc)
                packet_count += 1
        except Exception as e:
            print(f"\nRead error: {e}")
            break


# =============================================================================
# CONNECT & CONFIGURE
# =============================================================================

print(f"Sending OSC to {OSC_IP}:{OSC_PORT}\n")
osc_client = udp_client.SimpleUDPClient(OSC_IP, OSC_PORT)

sb = SifiBridge()
print(f"Connecting to {MY_DEVICE_MAC} ...")
while not sb.connect(MY_DEVICE_MAC):
    print("  not found yet, retrying...")
print("Connected!\n")

sb.set_filters(enable=True)
sb.configure_emg(state=True, fs=SAMPLE_RATES["emg_armband"], dc_notch=True, mains_notch=50, bandpass=True, flo=20, fhi=450)
sb.configure_imu(state=True, fs=SAMPLE_RATES["imu"])
sb.configure_sensors(emg=True, imu=True)
sb.set_low_latency_mode(on=False)  # Low-latency mode is not yet implemented with the current firmware version of the SiFi Band, but will be in a future update.
time.sleep(0.5)  # Give the device a moment to apply settings before starting the stream.

# Tell the receiver the nominal rates once upfront so it can size its buffers.
print("Announcing nominal sample rates:")
for signal, sr in SAMPLE_RATES.items():
    if ACTIVE_CHANNELS.get(signal):
        osc_client.send_message(f"/sifi/{signal}/sample_rate", float(sr))
        print(f"  /sifi/{signal}/sample_rate -> {sr} Hz")


# =============================================================================
# STREAM
#
# Console prints a live status line every 0.5 s:
#   elapsed | total packets | measured rate vs nominal for each active channel
#
# A healthy stream stays within a few % of nominal. A large gap indicates
# Bluetooth congestion or packet loss between the device and this computer.
#
# Press Ctrl+C (or wait for DURATION seconds) to stop cleanly.
# =============================================================================

try:
    sb.start()
    print("\nStreaming started. Press Ctrl+C to stop.\n")

    acq = threading.Thread(target=acquisition_thread, args=(sb, osc_client), daemon=True)
    acq.start()

    t_start = time.time()
    try:
        while DURATION is None or (time.time() - t_start) < DURATION:
            elapsed   = time.time() - t_start
            rates_str = "  |  ".join(
                f"{ch.upper()}: {rt.rate():6.1f} Hz (nominal {SAMPLE_RATES[ch]})"
                for ch, rt in rate_trackers.items()
            )
            print(f"\r  {elapsed:6.1f}s  |  packets: {packet_count:6d}  |  {rates_str}   ",
                  end="", flush=True)
            time.sleep(0.5)
    except KeyboardInterrupt:
        print("\n\nStopping...")

    stop_event.set()
    sb.stop()
    sb.disconnect()
    acq.join(timeout=2)
    print(f"Done. Total packets forwarded: {packet_count}")

finally:
    # Safety net: ensures the device is always released even on unexpected errors
    sb.stop()
    time.sleep(0.3)
    sb.disconnect()
    time.sleep(0.5)


Sending OSC to 127.0.0.1:9000

Connecting to C7:ED:F6:CF:B7:BE ...
Connected!

Announcing nominal sample rates:
  /sifi/emg_armband/sample_rate -> 1600 Hz
  /sifi/imu/sample_rate -> 50 Hz

Streaming started. Press Ctrl+C to stop.

    44.6s  |  packets:   5693  |  EMG_ARMBAND: 1591.1 Hz (nominal 1600)  |  IMU:   92.3 Hz (nominal 50)   

Stopping...
Done. Total packets forwarded: 5760
